# Dataset stats (current labels)


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/ and src/.")


PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"
LABELS_PATH = DATA_DIR / "labels.jsonl"
PLOTS_DIR = PROJECT_ROOT / "notebooks" / "data" / "dataset-stats"

LABEL_COLUMNS = ["trick", "execution_score", "key_foot", "person"]
AMBIGUITY_COLUMNS = ["ambiguous_score", "ambiguous_trick"]

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
def has_value(series: pd.Series) -> pd.Series:
    return series.notna() & series.astype("string").str.strip().ne("")


CLIP_RANGE_RE = re.compile(
    r"(\d{2})\.(\d{2})\.(\d{2})\.(\d{3})-(\d{2})\.(\d{2})\.(\d{2})\.(\d{3})"
)


def timestamp_parts_to_seconds(
    hours: str, minutes: str, seconds: str, millis: str
) -> float:
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds) + int(millis) / 1000


def parse_clip_duration(video: str):
    match = CLIP_RANGE_RE.search(Path(str(video)).name)
    if match is None:
        return pd.NA

    start = timestamp_parts_to_seconds(*match.groups()[:4])
    end = timestamp_parts_to_seconds(*match.groups()[4:])
    return end - start


def format_execution_score(series: pd.Series) -> pd.Series:
    scores = pd.to_numeric(series, errors="coerce").astype("Int64")
    return scores.astype("string")


def load_labels(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing labels file: {path}")

    df = pd.read_json(path, lines=True)

    for column in [*LABEL_COLUMNS, *AMBIGUITY_COLUMNS]:
        if column not in df.columns:
            df[column] = pd.NA

    for column in ["created_at", "updated_at"]:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce", utc=True)

    df["execution_score_label"] = format_execution_score(df["execution_score"])
    df["video_id"] = df["video"].map(
        lambda value: Path(str(value)).name if pd.notna(value) else pd.NA
    )
    df["clip_duration_seconds"] = df["video"].map(parse_clip_duration)
    df["has_ambiguous_score"] = has_value(df["ambiguous_score"])
    df["has_ambiguous_trick"] = has_value(df["ambiguous_trick"])
    return df


labels_df = load_labels(LABELS_PATH)
score_labels_df = labels_df.loc[has_value(labels_df["execution_score_label"])].copy()
complete_labels_df = labels_df.loc[
    has_value(labels_df["video"])
    & has_value(labels_df["trick"])
    & has_value(labels_df["execution_score_label"])
].copy()

summary_df = pd.DataFrame(
    [
        {"metric": "rows", "count": len(labels_df)},
        {"metric": "unique videos", "count": labels_df["video_id"].nunique()},
        {
            "metric": "rows with trick",
            "count": int(has_value(labels_df["trick"]).sum()),
        },
        {"metric": "rows with execution score", "count": len(score_labels_df)},
        {
            "metric": "rows with clip duration",
            "count": int(labels_df["clip_duration_seconds"].notna().sum()),
        },
        {"metric": "complete score rows", "count": len(complete_labels_df)},
        {
            "metric": "ambiguous score rows",
            "count": int(labels_df["has_ambiguous_score"].sum()),
        },
        {
            "metric": "ambiguous trick rows",
            "count": int(labels_df["has_ambiguous_trick"].sum()),
        },
    ]
)
display(summary_df)
display(labels_df.head())

In [ ]:
def save_plot(fig, name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path


def value_counts_table(
    frame: pd.DataFrame, column: str, order=None, include_missing: bool = False
) -> pd.DataFrame:
    series = frame[column]
    if include_missing:
        series = series.astype("string").fillna("(missing)")
        dropna = False
    else:
        dropna = True

    counts = series.value_counts(dropna=dropna)
    if order is not None:
        counts = counts.reindex(order, fill_value=0)

    counts_df = counts.rename_axis(column).reset_index(name="count")
    total = counts_df["count"].sum()
    counts_df["pct"] = (counts_df["count"] / total * 100).round(2) if total else 0
    return counts_df


def plot_count(
    frame: pd.DataFrame,
    column: str,
    name: str,
    *,
    order=None,
    include_missing=False,
    figsize=(8, 4),
    rotation=45,
):
    counts_df = value_counts_table(
        frame, column, order=order, include_missing=include_missing
    )

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=counts_df, x=column, y="count", order=counts_df[column].tolist(), ax=ax
    )
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.bar_label(ax.containers[0], padding=3)
    ax.tick_params(axis="x", rotation=rotation)
    save_plot(fig, name)
    plt.show()
    display(counts_df)
    return counts_df


rating_order = sorted(
    score_labels_df["execution_score_label"].dropna().unique(), key=int
)
trick_order = labels_df["trick"].value_counts().index.tolist()

## Missing and ambiguous labels


In [ ]:
missing_df = pd.DataFrame(
    {
        "column": LABEL_COLUMNS,
        "missing_count": [
            int((~has_value(labels_df[column])).sum()) for column in LABEL_COLUMNS
        ],
    }
)
missing_df["missing_pct"] = (missing_df["missing_count"] / len(labels_df) * 100).round(
    2
)

ambiguity_df = pd.DataFrame(
    [
        {
            "label": "execution_score",
            "rows_with_label": len(score_labels_df),
            "ambiguous_count": int(labels_df["has_ambiguous_score"].sum()),
            "ambiguous_pct_of_labeled": round(
                labels_df["has_ambiguous_score"].sum() / len(score_labels_df) * 100, 2
            )
            if len(score_labels_df)
            else 0,
        },
        {
            "label": "trick",
            "rows_with_label": int(has_value(labels_df["trick"]).sum()),
            "ambiguous_count": int(labels_df["has_ambiguous_trick"].sum()),
            "ambiguous_pct_of_labeled": (
                labels_df["has_ambiguous_trick"].mean() * 100
            ).round(2),
        },
    ]
)

display(missing_df)
display(ambiguity_df)

fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(data=ambiguity_df, x="label", y="ambiguous_count", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Ambiguous rows")
ax.bar_label(ax.containers[0], padding=3)
save_plot(fig, "ambiguous_label_counts")
plt.show()

## Trick distribution


In [ ]:
trick_counts = plot_count(
    labels_df,
    "trick",
    "trick_dist",
    order=trick_order,
    figsize=(9, 4),
    rotation=60,
)

## Ratings distribution


In [ ]:
score_counts = plot_count(
    score_labels_df,
    "execution_score_label",
    "ratings_dist",
    order=rating_order,
    figsize=(6, 4),
    rotation=0,
)

## Person distribution


In [ ]:
person_counts = plot_count(
    labels_df,
    "person",
    "person_dist",
    order=labels_df["person"].value_counts().index.tolist(),
    figsize=(7, 4),
    rotation=40,
)

## Key foot distribution


In [ ]:
key_foot_counts = plot_count(
    labels_df,
    "key_foot",
    "key_foot_dist",
    order=labels_df["key_foot"].value_counts().index.tolist(),
    figsize=(4, 4),
    rotation=0,
)

## Clip duration distribution


In [ ]:
clip_durations = labels_df["clip_duration_seconds"].dropna()
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(clip_durations, bins=30, kde=True, ax=ax)
ax.set_xlabel("Clip duration (seconds)")
ax.set_ylabel("Count")
save_plot(fig, "clip_duration_dist")
plt.show()

display(clip_durations.describe().to_frame(name="clip_duration_seconds"))

## Annotation lead time distribution


In [ ]:
lead_times = labels_df["lead_time"].dropna()
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(lead_times, bins=30, kde=True, ax=ax)
ax.set_xlabel("Annotation lead time (seconds)")
ax.set_ylabel("Count")
save_plot(fig, "annotation_lead_time_dist")
plt.show()

display(lead_times.describe().to_frame(name="lead_time_seconds"))

## Trick vs skater heatmap


In [ ]:
trick_person_counts = pd.crosstab(labels_df["trick"], labels_df["person"]).reindex(
    index=trick_order, fill_value=0
)

fig, ax = plt.subplots(figsize=(9, max(4, len(trick_person_counts) * 0.45)))
sns.heatmap(trick_person_counts, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_xlabel("Skater")
ax.set_ylabel("Trick")
save_plot(fig, "trick_skater_heatmap")
plt.show()

display(trick_person_counts)

## Trick vs execution score heatmap


In [ ]:
trick_score_counts = pd.crosstab(
    score_labels_df["trick"],
    score_labels_df["execution_score_label"],
).reindex(index=trick_order, columns=rating_order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, max(4, len(trick_score_counts) * 0.45)))
sns.heatmap(trick_score_counts, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_xlabel("Execution score")
ax.set_ylabel("Trick")
save_plot(fig, "trick_execution_score_heatmap")
plt.show()

display(trick_score_counts)